# Causal Inference in Practice
## Week 11 — Synthetic Control · Practice Notebook

> **Block III — Quasi-experimental designs**
>
> Build a synthetic version of the treated unit from a weighted blend of untreated ones.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · The setup — one treated unit and a donor pool

Synthetic control lives in a world where **exactly one unit is treated** at a known time and we have a panel of untreated *donor* units. We simulate that here. All units are driven by a few **shared latent factors** (so a weighted blend of donors can reproduce the treated unit), and we inject a **known additive treatment effect** into the treated unit's post-period. That ground truth is what our estimator must recover.

In [ ]:
from scipy.optimize import minimize

n_donors = 20
T, T0 = 30, 20          # 30 periods; treatment after period index 19
TRUE_EFFECT = 8.0       # known additive effect on the treated post-period

time = np.arange(T)
# three shared latent factors everyone loads on
factors = np.column_stack([
    np.sin(time / 3.0),
    np.linspace(0, 4, T),
    np.cos(time / 5.0) + time * 0.05,
])

def make_unit(loadings, level, noise=0.6):
    return level + factors @ loadings + RNG.normal(0, noise, size=T)

# Treated unit: loadings INSIDE the donors' spread so a convex blend can match it.
treated_loadings = np.array([2.0, 1.5, 2.5])
donor_loadings = RNG.normal(treated_loadings, 1.2, size=(n_donors, 3))
donor_levels   = RNG.normal(5.0, 1.5, size=n_donors)

Y = np.zeros((1 + n_donors, T))
Y[0] = make_unit(treated_loadings, 5.0)          # row 0 = treated
for j in range(n_donors):
    Y[1 + j] = make_unit(donor_loadings[j], donor_levels[j])

Y[0, T0:] += TRUE_EFFECT                          # inject the effect
treated, donors = Y[0], Y[1:]
print(f'panel: 1 treated + {n_donors} donors over {T} periods '
      f'({T0} pre, {T - T0} post)')

## 2 · Solve for convex donor weights

The estimator: choose weights `w` on the donors that **minimize pre-treatment RMSE** against the treated unit, subject to **`w ≥ 0`** and **`Σw = 1`**. We hand that constrained problem to `scipy.optimize.minimize` with SLSQP — box bounds for non-negativity and an equality constraint for the sum.

In [ ]:
def synth_weights(target_pre, donors_pre):
    """Non-negative weights summing to 1 that minimize pre-period MSE."""
    J = donors_pre.shape[0]
    loss = lambda w: np.mean((target_pre - w @ donors_pre) ** 2)
    cons = ({'type': 'eq', 'fun': lambda w: w.sum() - 1.0},)
    res = minimize(loss, np.full(J, 1.0 / J), method='SLSQP',
                   bounds=[(0.0, 1.0)] * J, constraints=cons,
                   options={'maxiter': 1000, 'ftol': 1e-12})
    return res.x

w = synth_weights(treated[:T0], donors[:, :T0])
pre_rmse = np.sqrt(np.mean((treated[:T0] - w @ donors[:, :T0]) ** 2))

print(f'weights sum to {w.sum():.3f}   (should be 1)')
print(f'min weight     {w.min():.4f}   (should be >= 0)')
print(f'donors used    {(w > 0.01).sum()} of {n_donors}   (convexity makes it sparse)')
print(f'pre-period RMSE {pre_rmse:.3f}')
assert abs(w.sum() - 1.0) < 1e-4, 'weights must sum to 1'
assert w.min() > -1e-6, 'weights must be non-negative'
assert pre_rmse < 1.0, 'a good blend should fit the pre-period tightly'

Note how **most weights are exactly zero** — the convexity constraints make the solution sparse, so the synthetic unit is a blend of just a few donors you could name and inspect.

## 3 · Build the synthetic control and plot it

Apply the weights across **all** periods. Before treatment the synthetic should hug the treated unit; after treatment it keeps tracking the counterfactual while the real treated unit jumps by the injected effect.

In [ ]:
synthetic = w @ donors                 # weights applied every period

fig, ax = plt.subplots()
ax.plot(time, treated,   label='treated unit', lw=2)
ax.plot(time, synthetic, '--', label='synthetic control', lw=2)
ax.axvline(T0 - 0.5, color='grey', ls=':', label='intervention')
ax.set_xlabel('period'); ax.set_ylabel('outcome')
ax.set_title('Treated vs. synthetic control')
ax.legend()
print('Pre-treatment: the two curves overlap. '
      'Post-treatment: a gap opens — that is the effect.')

## 4 · Estimate the gap and check it against the truth

The estimated effect is the average post-period gap, `treated − synthetic`. We compare it to the injected `TRUE_EFFECT` and assert recovery.

In [ ]:
gap = treated - synthetic
post_gap = gap[T0:].mean()
print(f'estimated effect = {post_gap:.3f}')
print(f'true effect      = {TRUE_EFFECT:.3f}')
print(f'pre-period gap   = {gap[:T0].mean():.3f}   (should be ~0 — good fit)')
assert abs(post_gap - TRUE_EFFECT) < 2.0, \
    'synthetic control should recover the injected effect'

### 🔧 Exercise 4.1 — convex weights beat equal weights

Difference-in-differences would weight every donor **equally**. Build the equal-weight synthetic (a plain average of all donors) and compare its **pre-period RMSE** to the fitted convex weights from Section 2. Which fits the treated unit's pre-period better, and why does that matter for the post-period gap?

Fill in the `# TODO`s below.

In [ ]:
# TODO: build the equal-weight synthetic and its pre-period RMSE.
equal_w = ...        # TODO: 1/n_donors on every donor
# synth_equal     = equal_w @ donors
# equal_pre_rmse  = np.sqrt(np.mean((treated[:T0] - synth_equal[:T0]) ** 2))
# print(equal_pre_rmse, pre_rmse)

### ✅ Solution 4.1

In [ ]:
equal_w = np.full(n_donors, 1.0 / n_donors)
synth_equal = equal_w @ donors
equal_pre_rmse = np.sqrt(np.mean((treated[:T0] - synth_equal[:T0]) ** 2))
print(f'equal-weight pre-RMSE = {equal_pre_rmse:.3f}')
print(f'convex-weight pre-RMSE = {pre_rmse:.3f}')
print('Fitted convex weights track the pre-period far better, so the '
      'post gap is a cleaner effect.')
assert equal_pre_rmse > pre_rmse, \
    'data-driven weights should fit the pre-period better than equal weights'

## 5 · Placebo-in-space inference (a permutation p-value)

With one treated unit there is no standard error. Instead we **pretend each donor was treated**: re-run the whole estimator with that donor as the target and the remaining units as its donor pool. The test statistic is the **ratio of post-period to pre-period RMSE**, which rewards a tight pre-period fit followed by a sharp post divergence. The treated unit's rank among all the ratios is an exact permutation p-value.

In [ ]:
def post_pre_ratio(g):
    pre  = np.sqrt(np.mean(g[:T0] ** 2))
    post = np.sqrt(np.mean(g[T0:] ** 2))
    return post / max(pre, 1e-8)

def placebo_gap(idx, Y):
    """Treat unit `idx` as if treated; donor pool = everyone else."""
    target, pool = Y[idx], np.delete(Y, idx, axis=0)
    wj = synth_weights(target[:T0], pool[:, :T0])
    return target - wj @ pool

treated_ratio = post_pre_ratio(gap)
placebo_ratios = np.array([post_pre_ratio(placebo_gap(j, Y))
                           for j in range(1, 1 + n_donors)])

all_ratios = np.append(placebo_ratios, treated_ratio)
p_value = np.mean(all_ratios >= treated_ratio)

print(f'treated post/pre RMSE ratio = {treated_ratio:.2f}')
print(f'placebo ratios: median {np.median(placebo_ratios):.2f}, max {placebo_ratios.max():.2f}')
print(f'permutation p-value = {p_value:.3f}')
assert p_value <= 0.10, 'the genuine effect should stand out from placebos'

In [ ]:
# Visualize: every placebo gap (grey) against the treated gap (red).
fig, ax = plt.subplots()
for j in range(1, 1 + n_donors):
    ax.plot(time, placebo_gap(j, Y), color='0.75', lw=1)
ax.plot(time, gap, color='crimson', lw=2.5, label='treated unit')
ax.axvline(T0 - 0.5, color='grey', ls=':')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('period'); ax.set_ylabel('gap (unit − synthetic)')
ax.set_title('In-space placebo gaps')
ax.legend()
print('The treated unit (red) is a clear outlier after the intervention.')

### 🔧 Exercise 5.1 — a permutation test on the raw post gap

Instead of the RMSE ratio, rank the **average post-period gap** itself. Compute each placebo's mean post-period gap, then the two-sided permutation p-value: the fraction of units (placebos plus the treated) whose `|post gap|` is at least as large as the treated unit's. Does the treated unit still come out significant?

Fill in the `# TODO`s below.

In [ ]:
# TODO: collect each placebo's mean post-period gap, then a two-sided p-value.
placebo_post_gaps = ...     # TODO: array of mean post gaps for donors 1..n
# treated_post = gap[T0:].mean()
# allg = np.append(placebo_post_gaps, treated_post)
# p_raw = np.mean(np.abs(allg) >= abs(treated_post))
# print(p_raw)

### ✅ Solution 5.1

In [ ]:
placebo_post_gaps = np.array(
    [placebo_gap(j, Y)[T0:].mean() for j in range(1, 1 + n_donors)])
treated_post = gap[T0:].mean()
allg = np.append(placebo_post_gaps, treated_post)
p_raw = np.mean(np.abs(allg) >= abs(treated_post))
print(f'treated mean post gap = {treated_post:.2f}')
print(f'placebo |post gap| median = {np.median(np.abs(placebo_post_gaps)):.2f}')
print(f'two-sided permutation p-value = {p_raw:.3f}')
assert p_raw <= 0.10, 'the treated gap should be extreme vs. placebos'

### 🔧 Exercise 5.2 — when synthetic control should refuse

Synthetic control fails when the treated unit lies **outside the donors' convex hull** — no convex blend can reach it. Construct a would-be treated unit with **extreme** factor loadings (far beyond any donor), fit weights to its pre-period, and show the pre-period RMSE is now huge. That large fit error is the method honestly refusing to build a counterfactual.

Fill in the `# TODO`s below.

In [ ]:
# TODO: an outlier unit with extreme loadings, then its pre-period RMSE.
bad_loadings = ...     # TODO: e.g. np.array([12.0, -10.0, 9.0])
# bad_unit = 5.0 + factors @ bad_loadings + RNG.normal(0, 0.6, size=T)
# w_bad    = synth_weights(bad_unit[:T0], donors[:, :T0])
# bad_rmse = np.sqrt(np.mean((bad_unit[:T0] - w_bad @ donors[:, :T0]) ** 2))
# print(bad_rmse, pre_rmse)

### ✅ Solution 5.2

In [ ]:
bad_loadings = np.array([12.0, -10.0, 9.0])     # far outside donor spread
bad_unit = 5.0 + factors @ bad_loadings + RNG.normal(0, 0.6, size=T)
w_bad = synth_weights(bad_unit[:T0], donors[:, :T0])
bad_rmse = np.sqrt(np.mean((bad_unit[:T0] - w_bad @ donors[:, :T0]) ** 2))
print(f'outlier pre-RMSE = {bad_rmse:.2f}')
print(f'good-fit pre-RMSE = {pre_rmse:.2f}')
print('No convex blend can match an out-of-hull unit — '
      'the huge RMSE is the method telling you not to trust the gap.')
assert bad_rmse > 3 * pre_rmse, \
    'an out-of-hull treated unit should fit far worse'

## 6 · Wrap-up & self-check

- **Synthetic control** builds the counterfactual as a *convex blend* of donors (`w ≥ 0`, `Σw = 1`) that matches the treated unit's pre-period.
- **Pre-treatment fit is the whole argument**: a small pre-period RMSE is what licenses reading the post-period gap as an effect — you recovered the injected truth here.
- **Convex weights interpolate, never extrapolate**: an out-of-hull treated unit fits badly, and that failure is informative (Exercise 5.2).
- **Inference is a permutation rank**, not a standard error: pretend each donor was treated and locate the real unit in the placebo distribution.
- **SC vs. DiD**: data-driven weights and enforced pre-period fit instead of equal weights and assumed parallel trends (Exercise 4.1).

**You're ready for Week 12** if you can solve for convex weights, explain why pre-treatment fit matters, and run an in-space placebo test from memory. Next week: **double / debiased machine learning** — Neyman-orthogonal scores and cross-fitting to plug ML into causal estimation.